# PRT661 – Housing Affordability Forecasting System
## Assessment 2

**Group:** Dan1-Theme2 · **Theme 2 — Predictive Analytics and Forecasting**

## Project Setup - Importing Libraries

In [40]:
import os
import re
import math
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.metrics import (
    mean_squared_error, mean_absolute_error, r2_score,
    classification_report, confusion_matrix, ConfusionMatrixDisplay
)
from xgboost import XGBRegressor

import joblib

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 60)

RANDOM_STATE = 19
np.random.seed(RANDOM_STATE)


In [ ]:
# Data directories
PROJECT_DIR = Path("..")
DATA_DIR = PROJECT_DIR / "data"
RAW_DIR = DATA_DIR / "raw"
PROCESSED_DIR = DATA_DIR / "processed"
OUTPUT_DIR = DATA_DIR / "output"

for d in [RAW_DIR, PROCESSED_DIR, OUTPUT_DIR]:
    d.mkdir(parents=True, exist_ok=True)

PROPERTIES_PATH = PROCESSED_DIR / "properties.csv"
POPULATION_PATH = PROCESSED_DIR / "population.csv"
HOUSEHOLD_INCOME_PATH = PROCESSED_DIR / "household_income.csv"

# Darwin CBD reference coordinates, used for the "distance from CBD" feature
DARWIN_CBD_LAT, DARWIN_CBD_LON = -12.4634, 130.8456


## 1. Loading Data
Loads the three raw CSVs and prints a quick health check (shape, dtypes, missing values) for
each — this doubles as the Veracity check referenced in Assessment 2, Section 3.2 (5 Vs).

In [ ]:
def load_csv(path, name):
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(
            f"Could not find file at {path}."
        )
    df = pd.read_csv(path)
    print(f"Loaded {name}: {df.shape[0]:,} rows x {df.shape[1]} columns")
    return df

properties_processed = load_csv(PROPERTIES_PATH, "Properties Dataset")
population_processed = load_csv(POPULATION_PATH, "Population Dataset")
household_income_processed = load_csv(HOUSEHOLD_INCOME_PATH, "Household and Income Dataset")


Loaded Properties Dataset: 15,423 rows x 18 columns
Loaded Population Dataset: 44 rows x 26 columns
Loaded Census Dataset: 305 rows x 23 columns


In [ ]:
def check_data(df, name):
    print(f"\n{'='*60}\n{name}\n{'='*60}")
    print(df.dtypes)
    missing = df.isna().sum()
    missing = missing[missing > 0].sort_values(ascending=False)
    if len(missing):
        print("\nMissing values:")
        print(missing)
    else:
        print("\nNo missing values.")
    display(df.head(2))

check_data(properties_processed, "Properties Dataset")
check_data(population_processed, "Population Dataset")
check_data(household_income_processed, "Household and Income Dataset")



Properties Dataset
property_id            int64
address                  str
url                      str
status                   str
sold_date                str
sold_date_iso            str
price_text               str
price_numeric        float64
price_undisclosed       bool
beds                   int64
baths                  int64
cars                   int64
land_area_m2             str
latitude             float64
longitude            float64
agent_name               str
agency_name              str
suburb                   str
dtype: object

Missing values:
agent_name       13273
agency_name      13273
land_area_m2      5800
price_text        5088
price_numeric     5088
sold_date          250
sold_date_iso      250
latitude           226
longitude          226
dtype: int64


,property_id,address,url,status,sold_date,sold_date_iso,price_text,price_numeric,price_undisclosed,beds,baths,cars,land_area_m2,latitude,longitude,agent_name,agency_name,suburb
0,11245030,"22 Caledonian Street, Anula NT 812",https://www.homely.com.au/homes/22-caledonian-...,Sold,15 Aug 2024,2024-08-15T00:00:00+00:00,"$540,000",540000.0,False,3,1,2,985m²,-12.393042,130.887771,Ella Carling,Real Estate Central,anula-nt-0812
1,11057061,"52 Wandie Crescent, Anula NT 812",https://www.homely.com.au/homes/52-wandie-cres...,Sold,4 Jul 2024,2024-07-04T00:00:00+00:00,"$560,000",560000.0,False,3,1,3,743m²,-12.394495,130.893049,Simon Watts,Real Estate Central,anula-nt-0812



Population Dataset
SA2 name      str
2001        int64
2002        int64
2003        int64
2004        int64
2005        int64
2006        int64
2007        int64
2008        int64
2009        int64
2010        int64
2011        int64
2012        int64
2013        int64
2014        int64
2015        int64
2016        int64
2017        int64
2018        int64
2019        int64
2020        int64
2021        int64
2022        int64
2023        int64
2024        int64
2025        int64
dtype: object

No missing values.


,SA2 name,2001,2002,2003,2004,2005,2006,2007,2008,2009,2010,2011,2012,2013,2014,2015,2016,2017,2018,2019,2020,2021,2022,2023,2024,2025
0,Darwin Airport,23,22,21,21,21,20,21,15,17,15,469,469,466,18,17,16,17,18,18,19,20,20,20,20,20
1,Darwin City,2140,2248,2320,2429,2661,2798,3052,3664,4352,4841,5053,5412,5862,6288,7056,7591,7917,7786,7633,7612,7578,7722,8077,8273,8597



Census Dataset
SAL_CODE_2021                                  str
suburb_locality                                str
census_year                                  int64
area_sqkm                                  float64
total_population                             int64
population_density_per_sqkm                float64
median_age_persons                           int64
median_personal_income_weekly_aud            int64
median_family_income_weekly_aud              int64
median_household_income_weekly_aud           int64
median_mortgage_repayment_monthly_aud        int64
median_rent_weekly_aud                       int64
average_persons_per_bedroom                float64
average_household_size                     float64
total_households_income_table                int64
known_income_households                      int64
pct_households_income_below_1000_weekly    float64
pct_households_income_2000plus_weekly      float64
one_person_households                        int64
total_household

,SAL_CODE_2021,suburb_locality,census_year,area_sqkm,total_population,population_density_per_sqkm,median_age_persons,median_personal_income_weekly_aud,median_family_income_weekly_aud,median_household_income_weekly_aud,median_mortgage_repayment_monthly_aud,median_rent_weekly_aud,average_persons_per_bedroom,average_household_size,total_households_income_table,known_income_households,pct_households_income_below_1000_weekly,pct_households_income_2000plus_weekly,one_person_households,total_households_size_table,pct_one_person_households,in_realestate_sample_suburbs,source
0,SAL70003,Alawa,2021,1.2399,2078,1675.941608,35,915,2186,2083,1777,340,0.9,2.8,679,612,0.245098,0.513072,118,679,0.173785,True,ABS 2021 Census General Community Profile Data...
1,SAL70014,Anula,2021,1.3178,2385,1809.834573,37,1034,2492,2269,2000,400,0.9,2.9,760,687,0.183406,0.567686,125,760,0.164474,True,ABS 2021 Census General Community Profile Data...


## 2. Data Cleaning

### Properties Dataset

**Important information from the data:**

- `suburb` is a **URL slug**, e.g. `"anula-nt-0812"` — not a clean name. Title-casing it directly
  would give `"Anula-Nt-0812"`, which would never match the ABS census's `"Alawa"`-style
  names. It must be parsed into suburb / state / postcode first.
- `land_area_m2` is **text with a unit suffix**, e.g. `"985m²"` — not a plain number.
- `price_undisclosed` is the **string** `"false"`/`"true"`, not `0`/`1`.
- `price_numeric` can be missing even when `price_text` (e.g. `"$540,000"`) is present, so a
  fallback parser is used.

Steps below: parse the suburb slug, coerce price/land-area to numeric, drop/flag undisclosed
prices, remove duplicates, and treat outliers in price and land area using an IQR filter
(consistent with the "define clean / trace errors / change thoughtfully" principles from the
Week 2 lecture).

#### Parsing

In [ ]:
def parse_suburb_slug(slug):
    """Turns a URL slug like 'coconut-grove-nt-0810' into a clean suburb name
    like 'Coconut Grove', by stripping a trailing postcode and state-code token."""
    if pd.isna(slug):
        return np.nan
    parts = str(slug).strip().lower().split("-")
    if parts and parts[-1].isdigit():        # trailing postcode, e.g. '0812'
        parts = parts[:-1]
    if parts and parts[-1].isalpha() and len(parts[-1]) <= 3:  # trailing state code, e.g. 'nt'
        parts = parts[:-1]
    return " ".join(p.capitalize() for p in parts) if parts else np.nan


def parse_land_area(value):
    """Strips unit suffixes like 'm²' from land area text and returns a float."""
    if pd.isna(value):
        return np.nan
    digits = re.sub(r"[^0-9.]", "", str(value))
    return float(digits) if digits else np.nan


def parse_bool_ish(value):
    """Handles 'true'/'false' strings (any case) as well as native booleans/0/1."""
    if isinstance(value, bool):
        return int(value)
    s = str(value).strip().lower()
    if s in ("true", "1", "yes"):
        return 1
    if s in ("false", "0", "no", "nan", ""):
        return 0
    return 0


properties = properties_processed.copy()

# Parse the suburb slug into a clean, human-readable suburb name for joining later
properties["suburb_raw"] = properties["suburb"]
properties["suburb"] = properties["suburb_raw"].apply(parse_suburb_slug)
print("Example suburb parsing:")
print(properties[["suburb_raw", "suburb"]].drop_duplicates().head(8).to_string(index=False))

# Parse dates
properties["sold_date_iso"] = pd.to_datetime(properties["sold_date_iso"], errors="coerce")

# Clean land area (strip 'm²' etc.)
if "land_area_m2" in properties.columns:
    properties["land_area_m2"] = properties["land_area_m2"].apply(parse_land_area)

# Make the undisclosed flag binary
if "price_undisclosed" in properties.columns:
    properties["price_undisclosed"] = properties["price_undisclosed"].apply(parse_bool_ish)
else:
    properties["price_undisclosed"] = 0

# Coerce price to numeric, like $540000 to 540000. We already have price_numeric, however, just in case.
properties["price_numeric"] = pd.to_numeric(properties["price_numeric"], errors="coerce")
if "price_text" in properties.columns:
    needs_fallback = properties["price_numeric"].isna() & properties["price_text"].notna()
    fallback_prices = (
        properties.loc[needs_fallback, "price_text"]
        .astype(str).str.replace(r"[^0-9.]", "", regex=True)
    )
    properties.loc[needs_fallback, "price_numeric"] = pd.to_numeric(fallback_prices, errors="coerce")
    print(f"Recovered {needs_fallback.sum()} prices from price_text via fallback parsing.")

# Remove duplicate listings
n_before = len(properties)
properties = properties.drop_duplicates()
print(f"Removed {n_before - len(properties)} duplicate records ({len(properties)} remain).")

# check missing
missing_pct = properties.isna().mean().sort_values(ascending=False) * 100
print("\nMissing % by column (Properties):")
print(missing_pct[missing_pct > 0])


Example suburb parsing:
           suburb_raw        suburb
        anula-nt-0812         Anula
      bayview-nt-0820       Bayview
coconut-grove-nt-0810 Coconut Grove
  darwin-city-nt-0800   Darwin City
   fannie-bay-nt-0820    Fannie Bay
      jingili-nt-0810       Jingili
       karama-nt-0812        Karama
   larrakeyah-nt-0820    Larrakeyah
Recovered 0 prices from price_text via fallback parsing.
Removed 0 duplicate records (15423 remain).

Missing % by column (Properties):
agency_name      86.059781
agent_name       86.059781
land_area_m2     37.606173
price_text       32.989691
price_numeric    32.989691
sold_date         1.620956
sold_date_iso     1.620956
longitude         1.465344
latitude          1.465344
dtype: float64


#### First Look

### ABS Census Household and Income Dataset

Standardises suburb names for joining, and reports which numeric columns contain missing
values before any imputation is applied (imputation happens after merging, in Section 6, so
we don't wrongly impute using statistics from suburbs that never appear in `properties.csv`).

In [ ]:
household_income = household_income_processed.copy()
household_income["SA2_name"] = household_income["SA2_name"].astype(str).str.strip().str.title()

# If multiple household income years exist per suburb, keep the most recent year available
if "year" in household_income.columns:
    household_income = household_income.sort_values("year").drop_duplicates(subset=["SA2_name"], keep="last")

missing_pct = household_income.isna().mean().sort_values(ascending=False) * 100
print("Missing % by column (ABS household income):")
print(missing_pct[missing_pct > 0])

household_income.head(2)


Missing % by column (ABS census):
pct_one_person_households                  14.426230
pct_households_income_2000plus_weekly      14.426230
pct_households_income_below_1000_weekly    14.426230
population_density_per_sqkm                 0.655738
dtype: float64


,SAL_CODE_2021,suburb_locality,census_year,area_sqkm,total_population,population_density_per_sqkm,median_age_persons,median_personal_income_weekly_aud,median_family_income_weekly_aud,median_household_income_weekly_aud,median_mortgage_repayment_monthly_aud,median_rent_weekly_aud,average_persons_per_bedroom,average_household_size,total_households_income_table,known_income_households,pct_households_income_below_1000_weekly,pct_households_income_2000plus_weekly,one_person_households,total_households_size_table,pct_one_person_households,in_realestate_sample_suburbs,source
0,SAL70003,Alawa,2021,1.2399,2078,1675.941608,35,915,2186,2083,1777,340,0.9,2.8,679,612,0.245098,0.513072,118,679,0.173785,True,ABS 2021 Census General Community Profile Data...
206,SAL70185,Milikapiti,2021,11.4786,414,36.067116,29,255,655,793,0,70,1.3,3.4,111,92,0.608696,0.065217,24,111,0.216216,False,ABS 2021 Census General Community Profile Data...


### ABS Census Population Dataset
`population.csv` stores one column per year (2001–2025). This reshapes it into a long/tidy
format (`suburb`, `year`, `population`) so it can be merged against each property's sale year.

In [73]:
population = population_processed.copy()
population["SA2_name"] = population["SA2_name"].astype(str).str.strip().str.title()

population.head()


KeyError: 'SA2_name'

## 3. Feature Engineering

### Distance from Darwin CBD

In [ ]:
def haversine_km(lat1, lon1, lat2, lon2):
    R = 6371.0  # Earth radius in km
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    dlat, dlon = lat2 - lat1, lon2 - lon1
    a = np.sin(dlat / 2) ** 2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2) ** 2
    return 2 * R * np.arcsin(np.sqrt(a))

properties["distance_from_cbd_km"] = haversine_km(
    properties["latitude"], properties["longitude"], DARWIN_CBD_LAT, DARWIN_CBD_LON
)

if "land_area_m2" in properties.columns:
    properties["price_per_sqm"] = properties["price_numeric"] / properties["land_area_m2"].replace(0, np.nan)

properties["sale_year"] = properties["sold_date_iso"].dt.year
properties["sale_month"] = properties["sold_date_iso"].dt.month

properties[["suburb", "distance_from_cbd_km", "price_per_sqm", "sale_year", "sale_month"]].describe(include="all").T
